## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...
            </span>
        </td>
    </tr>
</table>

In [1]:
# Start with imports - ask ChatGPT to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [2]:
# Always remember to do this!
load_dotenv(override=True)

True

In [3]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI
DeepSeek API Key not set (and this is optional)
Groq API Key not set (and this is optional)


In [28]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation. "
request += "Make the question short, but challenging"
messages = [{"role": "user", "content": request}]

In [29]:
messages

[{'role': 'user',
  'content': 'Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. Answer only with the question, no explanation. Make the question short, but challenging'}]

In [30]:
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=messages,
)
question = response.choices[0].message.content
print(question)


Devise and justify a decision rule for allocating one scarce life-saving dose among multiple patients when expected outcomes, fairness principles, and uncertainty conflict.


In [31]:
# 
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

## Note - update since the videos

I've updated the model names to use the latest models below, like GPT 5 and Claude Sonnet 4.5. It's worth noting that these models can be quite slow - like 1-2 minutes - but they do a great job! Feel free to switch them for faster models if you'd prefer, like the ones I use in the video.

In [32]:
# The API we know well
# I've updated this with the latest model, but it can take some time because it likes to think!
# Replace the model with gpt-4.1-mini if you'd prefer not to wait 1-2 mins
model_name = "gpt-5-nano"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

Here is a concrete, transparent decision rule you can implement when one scarce life-saving dose must be allocated among several patients, and you face conflicts among expected outcomes, fairness, and uncertainty.

Name of rule: Robust Fair Expected-Benefit (R-FEB) rule

Goals it seeks to balance
- Efficiency: allocate to the patient who yields the greatest expected health benefit (e.g., lives saved or QALYs gained) if given the dose.
- Robustness to uncertainty: avoid over-optimistic choices when prognosis is uncertain.
- Fairness: prevent systematic disadvantage to historically underserved or high-need groups, especially in repeated rounds of scarce-dose decisions.

Key quantities you need for each patient i
- B_i: estimated benefit of giving the dose to patient i (e.g., expected QALYs gained if successful).
- Uncertainty about B_i: characterize with a distribution or interval (e.g., Bayesian posterior for B_i, yielding a mean mu_i and a lower bound L_i such as a 5th percentile).
- Group membership g(i): which fairness-relevant group patient i belongs to (e.g., based on age, disability, socioeconomic status, or historical need).
- Fairness context: a mechanism to quantify “underserved-ness” of groups over time (e.g., small historical allocation rate, higher need, or a target share).

Decision rule (step-by-step)

Step 1 — estimate robust benefit scores
- For each patient i, obtain:
  - mu_i: posterior mean (or best estimate) of B_i.
  - L_i: a lower-confidence bound (e.g., 5th percentile) for B_i to capture uncertainty.
- Compute the robust efficiency benchmark:
  - mu_max = max_i mu_i.

Step 2 — form a near-optimal set under uncertainty
- Define a tolerance delta (a small nonnegative number you pre-specify to allow for uncertainty).
- Let C = { i : L_i >= mu_max - delta }.
  - This is the set of patients whose worst-case expected benefit is still within delta of the best estimated benefit. It protects against choosing someone who looks best only in the optimistic view.

Step 3 — incorporate fairness via a tie-breaking or scoring adjustment
- There are two practical options you can choose from (or use a combination as a policy):
  A) Fairness-tie-break (group-first)
     - If |C| = 1, allocate to that patient.
     - If |C| > 1, identify the most underserved group(s) among those in C (the group with the lowest historical allocation rate, or lowest target-share in the current policy window).
     - Prefer a patient from the most underserved group(s) within C. If several candidates are from underserved groups, break ties by the largest mu_i (or by random choice if mu_i are very close).
  B) Weighted fairness-adjusted score (continuous scoring)
     - For each i in C, compute a fairness boost F_i that depends on the group’s deficit score D_g(i) (a pre-specified fairness metric; e.g., difference between target allocation share and actual share, or an absolute priority for the most underserved group).
     - Define a composite score S_i = L_i + alpha * F_i, where alpha > 0 controls how strongly you weight fairness.
     - Allocate to i with the largest S_i (if multiple, break ties randomly).
- Notes on D_g and F_i:
  - D_g can be based on recent allocation history or need-based targets. For example, if group G has received only 60% of its target share in the past window, its deficit is 0.4, and F_i could be proportional to that deficit for any candidate in that group.
  - alpha is a policy knob. Larger alpha gives more weight to fairness; smaller alpha prioritizes robustness to uncertainty and efficiency.

Step 4 — make the allocation
- If using A (tie-break fairness): allocate to the candidate from the most underserved group within C, breaking any remaining ties by mu_i (or randomly).
- If using B (continuous scoring): allocate to the candidate with the largest S_i in C (breaking ties randomly).

Step 5 — (optional) robustness check and appeal
- After the decision, document the mu_i, L_i, and the chosen i, including how fairness was applied (which group was prioritized and why).
- In ongoing practice, periodically review the fairness weights and delta to ensure the policy behaves as intended over time.

Why this rule resolves conflicts between outcomes, fairness, and uncertainty

- Efficiency under uncertainty: By using mu_i to identify a near-maximum and L_i to define the near-optimal set, you protect against optimistic claims about prognosis. The mu_max - delta threshold ensures you’re not ignoring a credible near-best candidate because of uncertain prognoses.
- Fairness integrated without sacrificing too much efficiency: The rule uses a fairness tie-break (A) or a fairness-augmented score (B). In both cases, you give priority to underserved groups only when their potential benefit is not meaningfully worse than the top candidate. This respects efficiency most of the time but prevents systematic disadvantage over repeated decisions.
- Transparency and auditability: The rule is explicit about which numbers (mu_i, L_i, delta, and the fairness metric) determine the choice. It’s straightforward to explain to clinicians, patients, and ethics committees why a given patient was chosen.
- Flexibility: You can adapt delta, the fairness metric, and alpha to reflect policy priorities, ethical norms, or the dynamics of the specific setting (e.g., hospital, region, or emergency scenario).

A simple worked example (illustrative numbers)

- Four patients: A, B, C, D.
- Group membership: A and B are in underserved Group G1; C and D in Group G2.
- mu_i (mean estimated benefit) and L_i (lower bound of benefit):
  - A: mu_A = 8.0, L_A = 5.0
  - B: mu_B = 7.9, L_B = 4.9
  - C: mu_C = 7.6, L_C = 6.0
  - D: mu_D = 7.5, L_D = 5.5
- delta = 1.0 (allow for uncertainty)
- mu_max = 8.0 (A)
- Set C = {i : L_i >= mu_max - delta} = {A, B, C, D} since mu_max - delta = 7.0 and all L_i ≥ 4.9
- If you use rule A (underdog tie-break): among C, the most underserved group is G1 (A and B). So pick A or B. If you break tie by mu_i, A has higher mu, so allocate to A.
- If you used rule B with a fairness boost and alpha > 0: you would compute F_i for A and B to reflect G1’s deficit, possibly raising B’s S_i above A’s if you set alpha large enough; with a moderate alpha you might still pick A, but you’ve explicitly allowed fairness to influence the decision when results are close.

Practical notes and recommendations

- Choose delta and alpha thoughtfully:
  - delta should reflect how uncertain prognosis is; larger uncertainty justifies a larger delta, expanding the near-optimal set and increasing the chance that fairness factors influence the choice.
  - alpha controls how strongly you prioritize fairness; pick a value that aligns with your ethical and policy commitments and that can be justified to stakeholders.
- Define fairness metric carefully:
  - Use a clear, auditable measure of unfairness (e.g., recent allocation rates by group, or need-based population weights).
  - Ensure data on group membership and allocation history are accurate, up-to-date, and respect privacy.
- Tie-break rules matter:
  - When multiple candidates remain after applying fairness, have a predefined tie-break (randomization is often the fairest non-discriminatory method).
- Policy context matters:
  - In a single-shot crisis setting, this rule provides a principled compromise. In repeated rounds, track group shares over time to prevent drift toward chronic disadvantage.
- Limitations:
  - The quality of B_i depends on prognosis models, which can be biased or incomplete. Use robust data and update models as new information arrives.
  - Ethical acceptability depends on the specifics of the groups and the fairness targets chosen. Engage stakeholders when setting delta, alpha, and fairness metrics.
  - The rule does not account for all ethical considerations (e.g., non-medical factors, patient autonomy, consent). It should be part of a broader, transparent triage policy.

In sum
The Robust Fair Expected-Benefit rule gives a clear, implementable method to allocate one scarce life-saving dose by (1) prioritizing robust expectations of benefit, (2) acknowledging uncertainty, and (3) injecting a transparent fairness mechanism that protects historically underserved groups without forsaking overall efficiency.

In [33]:
# Anthropic has a slightly different API, and Max Tokens is required

model_name = "claude-sonnet-4-5"

claude = Anthropic()
response = claude.messages.create(model=model_name, messages=messages, max_tokens=1000)
answer = response.content[0].text

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

# Decision Rule for Scarce Life-Saving Resource Allocation

## Proposed Two-Stage Decision Framework

### Stage 1: Eligibility Screening (Clinical Filter)
Exclude only patients for whom treatment would be:
- **Medically futile** (< 5% survival probability even with treatment)
- **Physiologically impossible** (incompatible contraindications)

*Justification*: Respects medical reality without premature discrimination. Sets minimal threshold while preserving life-saving potential.

### Stage 2: Lexicographic Priority Rule

Apply criteria sequentially until tie-breaking achieved:

**Priority 1: Maximize "Life-Years-From-Point-of-Crisis" (LYFPC)**
- Calculate: (Survival probability with treatment) × (Remaining life expectancy from now)
- Not age alone, but life-saving potential from this intervention

**Priority 2: Fair Innings Adjustment** (if LYFPC within 20% margin)
- Prioritize those who've had fewer years of life
- Operationalize: If patients' LYFPC scores are close, choose younger patient

**Priority 3: Lottery** (if still tied)
- Random selection among remaining candidates

---

## Justification Structure

### Why This Framework?

**1. Consequentialist Foundation with Constraints**
- Primary goal: Save the most life (aggregate good)
- But constrained by fairness considerations that prevent pure utilitarianism

**2. Addresses the Three-Way Conflict**

| Principle | How Framework Handles It |
|-----------|-------------------------|
| **Expected Outcomes** | LYFPC maximizes life-saving impact while incorporating uncertainty via probability |
| **Fairness** | Fair innings adjustment + lottery ensure even poor-prognosis patients have consideration |
| **Uncertainty** | Explicit probability weighting; tie-breaking margin acknowledges measurement limits |

**3. Transparent and Defensible**
- Criteria can be explained to families
- Reduces bias compared to physician discretion
- Documentable and auditable

---

## Handling Key Objections

### "This discriminates by age"
**Response**: Not age per se, but life-saving potential. A 70-year-old with 85% survival probability and 15-year life expectancy could outrank a 30-year-old with 20% survival probability and organ failure. The framework asks: "How much life can we save *from this crisis*?"

### "Probabilities are uncertain"
**Response**: 
1. Use best available evidence (clinical scores, validated models)
2. The 20% equivalence margin acknowledges uncertainty
3. Sensitivity analysis: If outcome truly uncertain, fairness tiebreakers activate

### "What about social value (caregivers, essential workers)?"
**Response**: Deliberately excluded because:
- Impossibly subjective (whose life "matters more" is ethically dangerous)
- Slippery slope to discrimination
- If society wants to prioritize essential workers, create separate allocation pathway—don't contaminate medical triage

### "Equal lottery would be fairer"
**Response**: Pure lottery wastes life-saving capacity. If we can save 40 expected life-years vs. 5, failing to consider this difference treats identified lives as infinitely valuable while accepting many more statistical deaths. The framework respects the patient *in front of us* while acknowledging our obligation to all potential future patients.

---

## Worked Example

**Four patients, one dose:**

| Patient | Age | Survival Probability w/ Treatment | Life Expectancy If Survives | LYFPC |
|---------|-----|-----------------------------------|----------------------------|-------|
| A | 28 | 75% | 50 years | 37.5 |
| B | 45 | 90% | 35 years | 31.5 |
| C | 62 | 85% | 20 years | 17.0 |
| D | 72 | 60% | 12 years | 7.2 |

**Decision**: Patient A receives treatment (highest LYFPC)

**If A and B were within 20%**: Check fair innings → A is younger → A receives treatment

**If A and B were same age and within 20%**: Lottery between A and B

---

## Structural Advantages

1. **Lex

In [51]:
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-3.5-flash"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

To allocate a single, scarce, life-saving dose among multiple patients when expected outcomes, fairness principles, and clinical uncertainty conflict, we must reject simplistic single-metric solutions (like pure lotteries or pure utilitarian maximization). 

Instead, we must implement a **Multi-Stage Constrained Allocation Rule (MSCAR)**. This rule treats clinical utility as a coarse filter and procedural fairness as the ultimate selector, mediated by an acknowledgment of epistemic uncertainty.

---

### The Decision Rule: Multi-Stage Constrained Allocation (MSCAR)

```
[All Candidates]
       │
       ▼
┌────────────────────────────────────────┐
│ Step 1: Broad Prognostic Banding       │ -> Excludes futility; groups patients
└────────────────────────────────────────┘    into wide survival-probability tiers.
       │
       ▼
┌────────────────────────────────────────┐
│ Step 2: The Life-Cycle Adjustment      │ -> Prioritizes those who have not lived
└────────────────────────────────────────┘    through basic life stages (e.g., youth).
       │
       ▼
┌────────────────────────────────────────┐
│ Step 3: Weighted Lottery (Tie-Breaker) │ -> Blind random selection among equals
└────────────────────────────────────────┘    to ensure absolute moral fairness.
```

#### Step 1: Broad Prognostic Banding (Addressing Outcome Utility & Uncertainty)
Rather than ranking patients by precise, highly uncertain survival percentages (e.g., comparing a 72% chance to a 75% chance), clinicians group eligible patients into three broad **Prognostic Bands** based on their likelihood of surviving to hospital discharge *with* the treatment:
*   **Band A (High Utility):** >70% chance of short-term survival.
*   **Band B (Moderate Utility):** 30% to 70% chance of short-term survival.
*   **Band C (Low Utility/Futility):** <30% chance of short-term survival, or death is imminent regardless of treatment.

*Rule:* Allocation moves sequentially down the bands. If there are patients in Band A, the dose *cannot* be given to anyone in Band B or C. 

#### Step 2: The Life-Cycle Adjustment (Addressing Fairness & Intergenerational Justice)
If multiple patients reside within the highest active Prognostic Band (e.g., three patients in Band A), apply the **Life-Cycle Principle**. 
*   *Rule:* Prioritize patients who have not yet had the opportunity to live through the standard stages of life (childhood, adolescence, young adulthood). 
*   *Implementation:* Group candidates into two age brackets: **Under 30** (Priority) and **Over 30** (Standard). If a patient in Band A is under 30, they receive priority over those over 30 in the same band.

#### Step 3: The Equal-Dignity Lottery (Addressing Pure Egalitarianism & Bias)
If multiple patients remain tied within the same Prognostic Band and Life-Cycle category, the final allocation must be decided by a **blind, randomized lottery**.
*   *Rule:* All remaining tied patients have an equal chance. No clinical, socio-economic, or demographic data may be used to alter the odds.

---

### Justification of the Rule

The MSCAR model is justified by balancing three core bioethical frameworks: **Utilitarianism** (maximizing good), **Egalitarianism** (equal moral worth), and **Epistemic Humility** (acknowledging uncertainty).

#### 1. Why use "Broad Bands" instead of precise survival estimates?
*   **The Problem of Pseudo-Precision (Uncertainty):** Medical prognostication is not an exact science. Claiming Patient X has a 68% survival chance and Patient Y has a 64% chance is statistically noisy and ethically fragile.
*   **The Justification:** By grouping patients into broad bands (Step 1), we respect **epistemic humility**. We acknowledge that we cannot predict the future with decimal-point accuracy, yet we prevent the ethical catastrophe of "wasting" a dose on a patient who will almost certainly die (Band C) when a patient who will almost certainly live (Band A) is present. This satisfies the utilitarian demand to prevent waste without falling into the trap of over-precision.

#### 2. Why include the Life-Cycle Adjustment?
*   **The Problem of "Fair Innings" (Fairness vs. Age):** Treating an 8-year-old and an 80-year-old as having the exact same claim to a dose ignores the concept of a complete life.
*   **The Justification:** This is not ageism; it is the **"Fair Innings" argument**. The 80-year-old has had the opportunity to experience life's stages; the 8-year-old has not. Prioritizing the young within the same prognostic band maximizes the opportunity for all individuals to live a full life-cycle, which is a fundamental requirement of distributive justice.

#### 3. Why end with a Lottery?
*   **The Problem of Implicit Bias:** When clinicians try to make "micro-allocation" decisions between two similar patients (e.g., looking at "social worth," employment, or lifestyle factors), implicit biases regarding race, class, and gender inevitably creep in.
*   **The Justification:** A lottery (Step 3) reasserts **Kantian egalitarianism**—the belief that all human lives are of equal intrinsic value. Once we have established that both patients have a good chance of survival (Band A) and are in similar life stages, we must admit that we have no moral authority to declare one life "better" than the other. A lottery removes human bias, protects clinicians from moral injury, and treats both patients with equal dignity.

#### 4. Rejection of Alternative Rules
*   **Why not First-Come, First-Served (FCFS)?** FCFS is superficially fair but structurally unjust. It heavily favors wealthy, well-connected patients who have easier access to transportation, primary care, and major medical centers, thereby compounding systemic healthcare inequalities.
*   **Why not Maximize Total Life-Years?** Calculating total future life-years (QALYs) heavily discriminates against people with chronic disabilities who, even if cured of the acute illness, may have a shorter life expectancy. MSCAR focuses on *short-term survival to discharge* to avoid discriminating against the disabled.

### Conclusion
The **MSCAR** rule is a compromise of necessity. It acknowledges that in a tragedy of scarcity, we cannot achieve perfect utility and perfect fairness simultaneously. Instead, it uses **clinical utility to prevent tragedy (waste)**, **life-cycle adjustments to promote opportunity**, and **randomization to preserve human dignity in the face of uncertainty**.

In [37]:
# deepseek = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com/v1")
# model_name = "deepseek-chat"

# response = deepseek.chat.completions.create(model=model_name, messages=messages)
# answer = response.choices[0].message.content

# display(Markdown(answer))
# competitors.append(model_name)
# answers.append(answer)

In [38]:
# # Updated with the latest Open Source model from OpenAI

# groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
# model_name = "openai/gpt-oss-120b"

# response = groq.chat.completions.create(model=model_name, messages=messages)
# answer = response.choices[0].message.content

# display(Markdown(answer))
# competitors.append(model_name)
# answers.append(answer)


## For the next cell, we will use Ollama

Ollama runs a local web service that gives an OpenAI compatible endpoint,  
and runs models locally using high performance C++ code.

If you don't have Ollama, install it here by visiting https://ollama.com then pressing Download and following the instructions.

After it's installed, you should be able to visit here: http://localhost:11434 and see the message "Ollama is running"

You might need to restart Cursor (and maybe reboot). Then open a Terminal (control+\`) and run `ollama serve`

Useful Ollama commands (run these in the terminal, or with an exclamation mark in this notebook):

`ollama pull <model_name>` downloads a model locally  
`ollama ls` lists all the models you've downloaded  
`ollama rm <model_name>` deletes the specified model from your downloads

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Super important - ignore me at your peril!</h2>
            <span style="color:#ff7800;">The model called <b>llama3.3</b> is FAR too large for home computers - it's not intended for personal computing and will consume all your resources! Stick with the nicely sized <b>llama3.2</b> or <b>llama3.2:1b</b> and if you want larger, try llama3.1 or smaller variants of Qwen, Gemma, Phi or DeepSeek. See the <A href="https://ollama.com/models">the Ollama models page</a> for a full list of models and sizes.
            </span>
        </td>
    </tr>
</table>

In [39]:
#!ollama pull llama3.2

In [40]:
# ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
# model_name = "llama3.2"

# response = ollama.chat.completions.create(model=model_name, messages=messages)
# answer = response.choices[0].message.content

# display(Markdown(answer))
# competitors.append(model_name)
# answers.append(answer)

In [53]:
# So where are we?

print(competitors)
print(answers)


['gpt-5-nano', 'claude-sonnet-4-5', 'gemini-2.5-flash', 'gemini-3.5-flash']
['Here is a concrete, transparent decision rule you can implement when one scarce life-saving dose must be allocated among several patients, and you face conflicts among expected outcomes, fairness, and uncertainty.\n\nName of rule: Robust Fair Expected-Benefit (R-FEB) rule\n\nGoals it seeks to balance\n- Efficiency: allocate to the patient who yields the greatest expected health benefit (e.g., lives saved or QALYs gained) if given the dose.\n- Robustness to uncertainty: avoid over-optimistic choices when prognosis is uncertain.\n- Fairness: prevent systematic disadvantage to historically underserved or high-need groups, especially in repeated rounds of scarce-dose decisions.\n\nKey quantities you need for each patient i\n- B_i: estimated benefit of giving the dose to patient i (e.g., expected QALYs gained if successful).\n- Uncertainty about B_i: characterize with a distribution or interval (e.g., Bayesian pos

In [54]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


Competitor: gpt-5-nano

Here is a concrete, transparent decision rule you can implement when one scarce life-saving dose must be allocated among several patients, and you face conflicts among expected outcomes, fairness, and uncertainty.

Name of rule: Robust Fair Expected-Benefit (R-FEB) rule

Goals it seeks to balance
- Efficiency: allocate to the patient who yields the greatest expected health benefit (e.g., lives saved or QALYs gained) if given the dose.
- Robustness to uncertainty: avoid over-optimistic choices when prognosis is uncertain.
- Fairness: prevent systematic disadvantage to historically underserved or high-need groups, especially in repeated rounds of scarce-dose decisions.

Key quantities you need for each patient i
- B_i: estimated benefit of giving the dose to patient i (e.g., expected QALYs gained if successful).
- Uncertainty about B_i: characterize with a distribution or interval (e.g., Bayesian posterior for B_i, yielding a mean mu_i and a lower bound L_i such a

In [55]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [56]:
print(together)

# Response from competitor 1

Here is a concrete, transparent decision rule you can implement when one scarce life-saving dose must be allocated among several patients, and you face conflicts among expected outcomes, fairness, and uncertainty.

Name of rule: Robust Fair Expected-Benefit (R-FEB) rule

Goals it seeks to balance
- Efficiency: allocate to the patient who yields the greatest expected health benefit (e.g., lives saved or QALYs gained) if given the dose.
- Robustness to uncertainty: avoid over-optimistic choices when prognosis is uncertain.
- Fairness: prevent systematic disadvantage to historically underserved or high-need groups, especially in repeated rounds of scarce-dose decisions.

Key quantities you need for each patient i
- B_i: estimated benefit of giving the dose to patient i (e.g., expected QALYs gained if successful).
- Uncertainty about B_i: characterize with a distribution or interval (e.g., Bayesian posterior for B_i, yielding a mean mu_i and a lower bound L_i 

In [57]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [58]:
print(judge)

You are judging a competition between 4 competitors.
Each model has been given this question:

Devise and justify a decision rule for allocating one scarce life-saving dose among multiple patients when expected outcomes, fairness principles, and uncertainty conflict.

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}

Here are the responses from each competitor:

# Response from competitor 1

Here is a concrete, transparent decision rule you can implement when one scarce life-saving dose must be allocated among several patients, and you face conflicts among expected outcomes, fairness, and uncertainty.

Name of rule: Robust Fair Expected-Benefit (R-FEB) rule

Goals it seeks to balance
- Efficiency: allocate to the patient who yields the greatest expect

In [59]:
judge_messages = [{"role": "user", "content": judge}]

In [60]:
# Judgement time!

openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)


{"results": ["1", "3", "4", "2"]}


In [61]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

Rank 1: gpt-5-nano
Rank 2: gemini-2.5-flash
Rank 3: gemini-3.5-flash
Rank 4: claude-sonnet-4-5


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>